# node2vec — Scalable Feature Learning for Networks (2016)

_Papers · Graph Neural Networks_

**Bias the random walk with two knobs (return p, in-out q) so it interpolates between breadth-first and depth-first exploration, then run skip-gram on the walks to learn node embeddings.**

---

This notebook is a practice scaffold for the **node2vec — Scalable Feature Learning for Networks (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

random.seed(0); torch.manual_seed(0)

# --- 0. Worked example: one biased step. Walk went t->v; candidates are t (d=0), a (d=1), b (d=2). ---
def transition_probs(p, q, w=(1., 1., 1.)):
    # weights for (back d=0, local d=1, outward d=2)
    alpha = [1.0 / p, 1.0, 1.0 / q]
    pi = [a * wi for a, wi in zip(alpha, w)]
    Z = sum(pi)
    return [round(x / Z, 4) for x in pi]

print("p=2, q=0.5 ->", transition_probs(2, 0.5))   # [0.1429, 0.2857, 0.5714]  outward most likely (DFS-like)
print("p=2, q=2.0 ->", transition_probs(2, 2.0))   # [0.25,   0.5,    0.25]     local > outward (BFS-ward)
print("p=q=1 (DeepWalk) ->", transition_probs(1, 1))  # [0.3333,0.3333,0.3333]  uniform = unbiased walk


# --- 1. A toy graph: two dense communities joined by a single bridge. ---
# Community A = nodes 0..5, Community B = nodes 6..11. Each has a hub (0, 6) wired to its members
# plus a ring; one bridge edge (0,6) connects the communities.
def make_graph():
    edges = set()
    def add(u, v): edges.add((min(u, v), max(u, v)))
    for comm in ([0,1,2,3,4,5], [6,7,8,9,10,11]):
        hub = comm[0]
        for u in comm[1:]:
            add(hub, u)                       # hub connects to everyone in its community
        for i in range(1, len(comm)):         # plus a ring so the community is dense
            add(comm[i], comm[(i % (len(comm)-1)) + 1])
    add(0, 6)                                  # the single bridge between the two hubs
    N = 12
    adj = [[] for _ in range(N)]
    for u, v in edges:
        adj[u].append(v); adj[v].append(u)
    return N, [sorted(set(a)) for a in adj]

N, adj = make_graph()
nbr_set = [set(a) for a in adj]

# --- 2. The 2nd-order BIASED transition (the heart of node2vec). ---
def d_tx(t, x):
    if x == t: return 0                        # stepping straight back
    if x in nbr_set[t]: return 1               # x is also a neighbor of t (local)
    return 2                                    # x is one hop farther out

def biased_next(t, v, p, q):
    cands = adj[v]
    w = []
    for x in cands:
        d = d_tx(t, x)
        w.append(1.0/p if d == 0 else (1.0 if d == 1 else 1.0/q))
    s = sum(w)
    r, acc = random.random() * s, 0.0
    for x, wi in zip(cands, w):                 # sample proportional to the biased weights
        acc += wi
        if r <= acc: return x
    return cands[-1]

def walk(start, length, p, q):
    path = [start]
    cur = start; prev = start
    for _ in range(length - 1):
        nxt = biased_next(prev, cur, p, q) if len(path) > 1 else random.choice(adj[cur])
        path.append(nxt); prev, cur = cur, nxt
    return path

def sample_walks(p, q, num_walks=20, length=20):
    walks = []
    for _ in range(num_walks):
        for start in range(N):
            walks.append(walk(start, length, p, q))
    return walks

# --- 3. Skip-gram with negative sampling over the walks. ---
def train_embeddings(walks, dim=8, window=2, negatives=5, epochs=60, lr=0.05):
    torch.manual_seed(0)
    emb = nn.Embedding(N, dim); ctx = nn.Embedding(N, dim)
    nn.init.normal_(emb.weight, std=0.1); nn.init.normal_(ctx.weight, std=0.1)
    opt = torch.optim.Adam(list(emb.parameters()) + list(ctx.parameters()), lr=lr)
    # build (center, context) positive pairs from sliding windows
    pairs = []
    for wlk in walks:
        for i, c in enumerate(wlk):
            for j in range(max(0, i-window), min(len(wlk), i+window+1)):
                if j != i: pairs.append((c, wlk[j]))
    pairs = torch.tensor(pairs)
    for _ in range(epochs):
        perm = pairs[torch.randperm(len(pairs))]
        centers, contexts = perm[:, 0], perm[:, 1]
        negs = torch.randint(0, N, (len(perm), negatives))
        opt.zero_grad()
        ce, co = emb(centers), ctx(contexts)
        pos = F.logsigmoid((ce * co).sum(1))                         # pull center -> true context
        neg = F.logsigmoid(-(emb(centers).unsqueeze(1) * ctx(negs)).sum(2)).sum(1)  # push from negatives
        loss = -(pos + neg).mean()
        loss.backward(); opt.step()
    return F.normalize(emb.weight.detach(), dim=1)

import statistics, itertools

# --- 4a. q CHANGES THE WALK: DFS-like roams (more distinct nodes), BFS-like hugs the neighborhood. ---
def mean_distinct(p, q, trials=500, length=20):
    return statistics.mean(len(set(walk(random.randrange(N), length, p, q))) for _ in range(trials))

random.seed(0); dfs = mean_distinct(1, 0.25)
random.seed(0); bfs = mean_distinct(1, 4.0)
print(f"distinct nodes / length-20 walk:  DFS-like q=0.25 = {dfs:.2f}   BFS-like q=4 = {bfs:.2f}")
# DFS-like ~8.0 (roams outward -> homophily);  BFS-like ~6.1 (stays local -> structural equivalence).

# --- 4b. Skip-gram on the biased walks recovers the two communities (homophily read-out). ---
A, B = list(range(0, 6)), list(range(6, 12))
intra = list(itertools.combinations(A, 2)) + list(itertools.combinations(B, 2))
inter = [(i, j) for i in A for j in B]
def avg_cos(z, pairs):
    return statistics.mean(float((z[i] * z[j]).sum()) for i, j in pairs)

random.seed(0)
z = train_embeddings(sample_walks(1, 0.25))      # DFS-like walks -> homophily
print(f"DFS-like embedding:  intra-community cos = {avg_cos(z, intra):+.3f}   "
      f"inter-community cos = {avg_cos(z, inter):+.3f}")

# Our small run (NOT the paper's numbers): the in-out parameter q visibly changes the walk --
# DFS-like (q<1) explores ~8 distinct nodes per walk vs ~6 for BFS-like (q>1) -- and skip-gram on the
# biased walks recovers the community structure (intra cos ~0.9 >> inter cos ~0.2), i.e. homophily.

## Visualize the data & results

_Does the in-out parameter q actually change the walk — does a DFS-like walk (q1) that hugs the immediate neighborhood?_

In [ ]:
import random, statistics

# Toy graph: two dense communities (0-5, 6-11) joined by a single bridge edge (0,6).
def make_graph():
    edges = set()
    def add(u, v): edges.add((min(u, v), max(u, v)))
    for comm in ([0,1,2,3,4,5], [6,7,8,9,10,11]):
        hub = comm[0]
        for u in comm[1:]: add(hub, u)
        for i in range(1, len(comm)): add(comm[i], comm[(i % (len(comm)-1)) + 1])
    add(0, 6)
    adj = [[] for _ in range(12)]
    for u, v in edges: adj[u].append(v); adj[v].append(u)
    return [sorted(set(a)) for a in adj]

adj = make_graph(); N = 12; nbr = [set(a) for a in adj]

# The 2nd-order biased transition: weight depends on distance from the PREVIOUS node t.
def d_tx(t, x): return 0 if x == t else (1 if x in nbr[t] else 2)
def nxt(t, v, p, q):
    c = adj[v]; w = [1/p if d_tx(t,x)==0 else (1 if d_tx(t,x)==1 else 1/q) for x in c]
    s = sum(w); r = random.random()*s; acc = 0
    for x, wi in zip(c, w):
        acc += wi
        if r <= acc: return x
    return c[-1]
def walk(s, L, p, q):
    path=[s]; cur=s; prev=s
    for _ in range(L-1):
        n = nxt(prev,cur,p,q) if len(path)>1 else random.choice(adj[cur])
        path.append(n); prev,cur=cur,n
    return path

def mean_distinct(p, q, trials=500, length=20):
    return statistics.mean(len(set(walk(random.randrange(N), length, p, q))) for _ in range(trials))

random.seed(0); dfs = mean_distinct(1, 0.25)
random.seed(0); bfs = mean_distinct(1, 4.0)
print("DFS-like q=0.25 distinct/walk:", round(dfs, 2))   # ~8.0  (roams outward -> homophily)
print("BFS-like q=4    distinct/walk:", round(bfs, 2))   # ~6.1  (stays local -> structural equivalence)
# Our small run, not the paper's number. q interpolates between BFS and DFS exploration (&sect;3.2.2).

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Recompute the transition with different knobs. Same step $t\to v$ with candidates $t$
            ($d{=}0$), $a$ ($d{=}1$), $b$ ($d{=}2$), all edge weights $1$. Now use $p=0.25$, $q=4$. Give the
            unnormalized weights and the normalized probabilities, and say which behavior this encodes.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Biases: $\alpha(t)=1/p=1/0.25=4$; $\alpha(a)=1$; $\alpha(b)=1/q=1/4=0.25$. — _Plug $p,q$ into the three cases of $\alpha_{pq}$._
- Unnormalized $\pi$ (all $w{=}1$): $\pi_t=4,\ \pi_a=1,\ \pi_b=0.25$. Sum $=5.25$. — _$\pi_{vx}=\alpha\,w_{vx}$, then sum for normalization._
- Normalize: $P(t)=4/5.25\approx 0.762$, $P(a)=1/5.25\approx 0.190$, $P(b)=0.25/5.25\approx 0.048$. — _Divide each $\pi$ by the sum._

**Answer:** Weights $\pi=(4,\,1,\,0.25)$ give $P(t)\approx 0.762,\ P(a)\approx 0.190,\
                 P(b)\approx 0.048$. The small $p=0.25$ makes returning to $t$ overwhelmingly likely and
                 the large $q=4$ makes the outward step $b$ almost never chosen &mdash; the walk stays
                 glued to the start node's immediate neighborhood. This is a strongly BFS-like regime,
                 biasing the embedding toward structural equivalence (role similarity).

</details>

**Problem 2.** Show that $p=q=1$ reduces node2vec to an unbiased random walk (DeepWalk, paper-deepwalk).
            What are the three biases, and what is $P(b)$ for the example graph (candidates $t,a,b$)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set $p=q=1$: $\alpha(t)=1/p=1$, $\alpha(a)=1$, $\alpha(b)=1/q=1$. — _Every case of $\alpha_{pq}$ becomes $1$._
- Unnormalized weights all equal the edge weight ($1$ here); sum $=3$. — _$\pi_{vx}=1\cdot w_{vx}$, so the bias has no effect._
- $P(b)=1/3\approx 0.333$, same as $P(t)$ and $P(a)$. — _Uniform over neighbors = an unbiased 1st-order walk._

**Answer:** With $p=q=1$ all three biases equal $1$, so $\pi_{vx}=w_{vx}$ and the next node is chosen
                 uniformly among neighbors ($P(t)=P(a)=P(b)=1/3$). The dependence on the previous node
                 $t$ disappears &mdash; the walk is a plain unbiased random walk, which is exactly
                 DeepWalk. node2vec is its strict generalization.

</details>

**Problem 3.** Ablation on the walk itself. On a graph with two dense communities, you run length-$20$
            walks twice from every node: once DFS-like ($p=1,q=0.25$) and once BFS-like ($p=1,q=4$). Predict
            which regime visits more distinct nodes per walk, and tie that back to homophily vs
            structural equivalence (&sect;3.1).

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- DFS-like ($q\lt 1$): the outward weight $1/q=4$ is large, so each step tends to push to a farther node. — _The walk keeps moving outward, so a single walk of fixed length covers more of the graph &mdash; more distinct nodes._
- BFS-like ($q\gt 1$): the outward weight $1/q=0.25$ is small, so the walk re-samples the immediate neighborhood. — _It revisits the same local nodes, so a fixed-length walk touches fewer distinct nodes &mdash; it characterizes a node by its local wiring (role)._
- Count mean distinct nodes per walk in each regime; change only $q$. — _Isolating $q$ attributes any difference to the BFS/DFS exploration bias &mdash; the exact knob the paper introduces._

**Answer:** The DFS-like walk ($q=0.25$) visits more distinct nodes per walk &mdash; it
                 roams outward across the community, which is why its co-occurrence statistics reflect
                 homophily (community membership). The BFS-like walk ($q=4$) visits fewer
                 distinct nodes &mdash; it hugs the immediate neighborhood, characterizing each node by its
                 local wiring pattern, which is the path to structural equivalence (role similarity).
                 The CODEVIZ panel measures exactly this distinct-nodes-per-walk gap on a toy graph &mdash; a
                 direct, robust read-out of the $q$ knob interpolating between BFS and DFS.

</details>